In [3]:
from pathlib import Path
import sys


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Allow the notebook to import modules from the project root.
sys.path.append("..")


from config import CONFIG, REQUIRED_COLUMNS
from src.data_processing import (
    load_csv_chunks,
    prepare_chunk,
)
from src.analysis import CustomsAnalyzer
from src.numpy_analysis import run_numpy_comparison
from src.visualization import (
    create_bar_plot,
    create_heatmap,
)
from src.validation import (
    create_validation_results,
    save_validation_results,
)
CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)

In [4]:
# Inspect the full dataset before filtering.
raw_row_count = 0
missing_counts = pd.Series(
    0,
    index=sorted(REQUIRED_COLUMNS),
    dtype="int64",
)

observed_types = {
    column: set()
    for column in sorted(REQUIRED_COLUMNS)
}

for chunk in load_csv_chunks(
    CONFIG["input_path"],
    REQUIRED_COLUMNS,
    CONFIG["chunksize"],
):
    raw_row_count += len(chunk)

    missing_counts = missing_counts.add(
        chunk.isna().sum(),
        fill_value=0,
    ).astype("int64")

    for column in observed_types:
        observed_types[column].add(str(chunk[column].dtype))

inspection = pd.DataFrame(
    {
        "column": sorted(REQUIRED_COLUMNS),
        "observed_data_types": [
            ", ".join(sorted(observed_types[column]))
            for column in sorted(REQUIRED_COLUMNS)
        ],
        "missing_values": [
            int(missing_counts[column])
            for column in sorted(REQUIRED_COLUMNS)
        ],
    }
)

print(f"Raw row count: {raw_row_count:,}")
display(inspection)


Raw row count: 2,236,612


,column,observed_data_types,missing_values
0,countryorigin_iso3,object,0
1,dutiablevaluephp,int64,0
2,tq,object,0


In [5]:
chunks = load_csv_chunks(
    CONFIG["input_path"],
    REQUIRED_COLUMNS,
    CONFIG["chunksize"],
)

first_chunk = next(chunks)

print("First chunk rows:", len(first_chunk))
print("Columns:")
print(first_chunk.columns.tolist())

First chunk rows: 100000
Columns:
['tq', 'dutiablevaluephp', 'countryorigin_iso3']


In [6]:
analyzer = CustomsAnalyzer(
    output_dir=CONFIG["output_dir"],
    group_columns=CONFIG["group_columns"],
    measure_column=CONFIG["measure_column"],
)

chunks = load_csv_chunks(
    CONFIG["input_path"],
    REQUIRED_COLUMNS,
    CONFIG["chunksize"],
)

for chunk in chunks:
    selected = prepare_chunk(
        chunk,
        CONFIG["minimum_dutiable_value_php"],
        CONFIG["high_value_threshold_million_php"],
    )

    analyzer.add_chunk(
        selected,
        len(chunk),
        len(selected),
    )

data = analyzer.combine_chunks()

print("Selected rows:", len(data))
print()
print(data.head())

Selected rows: 2236612

       tq  dutiablevaluephp countryorigin_iso3  dutiablevalue_million_php  \
0  2015q1          90088007                MYS                  90.088007   
1  2015q1          15742304                CHN                  15.742304   
2  2015q1          52605886                CHN                  52.605886   
3  2015q1          34181694                CHN                  34.181694   
4  2015q1           7522344                KOR                   7.522344   

  value_band  
0   Standard  
1   Standard  
2   Standard  
3   Standard  
4   Standard  


In [7]:
data.loc[
    (
        data["tq"].notna()
        & (data["dutiablevaluephp"] > 0)
    ),
    [
        "tq",
        "countryorigin_iso3",
        "dutiablevaluephp",
        "dutiablevalue_million_php",
        "value_band",
    ],
].head(10)

,tq,countryorigin_iso3,dutiablevaluephp,dutiablevalue_million_php,value_band
0,2015q1,MYS,90088007,90.088007,Standard
1,2015q1,CHN,15742304,15.742304,Standard
2,2015q1,CHN,52605886,52.605886,Standard
3,2015q1,CHN,34181694,34.181694,Standard
4,2015q1,KOR,7522344,7.522344,Standard
5,2015q1,KOR,937401,0.937401,Standard
6,2015q1,KOR,454267,0.454267,Standard
7,2015q1,KOR,437421,0.437421,Standard
8,2015q1,CHN,9536863,9.536863,Standard
9,2015q1,CHN,123198882,123.198882,High


In [8]:
grouped = analyzer.create_grouped_summary(data)

grouped.head(10)

,countryorigin_iso3,row_count,valid_measure_count,measure_sum,measure_mean
0,AFG,5,5,588546,1.177092e+05
1,AGO,4,4,11752842,2.938210e+06
2,AIA,26,26,6120010,2.353850e+05
3,ALB,33,33,4355770,1.319930e+05
4,AND,8,8,16228840,2.028605e+06
5,ANT,81,81,393863146,4.862508e+06
6,ARE,2851,2851,29645763789,1.039837e+07
7,ARG,1089,1089,13213323494,1.213345e+07
8,ARM,5,5,998542,1.997084e+05
9,ASM,14,14,2575220,1.839443e+05


In [9]:
print("Number of country groups:", len(grouped))
print("Total grouped rows:", grouped["row_count"].sum())
print("Total grouped measure:", grouped["measure_sum"].sum())

Number of country groups: 197
Total grouped rows: 2236612
Total grouped measure: 3587267375257


In [10]:
grouped.to_csv(
    CONFIG["output_dir"] / "grouped.csv",
    index=False,
)

print("grouped.csv saved.")

grouped.csv saved.


In [11]:
grouped_two = analyzer.create_two_category_summary(data)

grouped_two.head(10)

,countryorigin_iso3,tq,row_count,measure_sum
0,AFG,2015q3,3,567713
1,AFG,2015q4,2,20833
2,AGO,2015q3,4,11752842
3,AIA,2015q3,9,1429763
4,AIA,2015q4,17,4690247
5,ALB,2015q2,1,138212
6,ALB,2015q3,16,947695
7,ALB,2015q4,16,3269863
8,AND,2015q2,3,1551437
9,AND,2015q3,3,4748724


In [12]:
print("Number of country-quarter groups:", len(grouped_two))
print("Total grouped rows:", grouped_two["row_count"].sum())
print("Total grouped measure:", grouped_two["measure_sum"].sum())

Number of country-quarter groups: 628
Total grouped rows: 2236612
Total grouped measure: 3587267375257


In [13]:
grouped_two.to_csv(
    CONFIG["output_dir"] / "grouped_two.csv",
    index=False,
)

print("grouped_two.csv saved.")

grouped_two.csv saved.


In [14]:
pivot = analyzer.create_pivot(grouped_two)

pivot.head(10)

tq,countryorigin_iso3,2015q1,2015q2,2015q3,2015q4,Total
0,AFG,NaN,NaN,5.677130e+05,2.083300e+04,588546
1,AGO,NaN,NaN,1.175284e+07,NaN,11752842
2,AIA,NaN,NaN,1.429763e+06,4.690247e+06,6120010
3,ALB,NaN,1.382120e+05,9.476950e+05,3.269863e+06,4355770
4,AND,NaN,1.551437e+06,4.748724e+06,9.928679e+06,16228840
5,ANT,4.093564e+07,3.245682e+07,2.156041e+08,1.048666e+08,393863146
6,ARE,1.143155e+10,5.646697e+09,8.342975e+09,4.224541e+09,29645763789
7,ARG,2.034266e+09,8.810729e+08,3.759537e+09,6.538448e+09,13213323494
8,ARM,NaN,NaN,2.677720e+05,7.307700e+05,998542
9,ASM,3.325130e+05,NaN,6.254110e+05,1.617296e+06,2575220


In [15]:
print("Pivot grand total:", pivot.loc[pivot["countryorigin_iso3"] == "Total", "Total"].iloc[0])

Pivot grand total: 3587267375257


In [16]:
pivot.to_csv(
    CONFIG["output_dir"] / "pivot.csv",
    index=False,
)

print("pivot.csv saved.")

pivot.csv saved.


In [17]:
top10 = analyzer.create_top10(grouped)

top10

,countryorigin_iso3,row_count,valid_measure_count,measure_sum,measure_mean
0,CHN,589626,589626,651605135422,1.105116e+06
1,JPN,352375,352375,304562507423,8.643136e+05
2,USA,209259,209259,291521135474,1.393112e+06
3,KOR,112063,112063,266807274024,2.380869e+06
4,THA,86863,86863,252139451874,2.902726e+06
5,TWN,93071,93071,244973287949,2.632112e+06
6,SGP,208349,208349,230099848496,1.104396e+06
7,IDN,35819,35819,162933845447,4.548811e+06
8,MYS,64880,64880,161741895330,2.492939e+06
9,SAU,602,602,158896912371,2.639484e+08


In [18]:
print("Number of rows in top10:", len(top10))
print("Top 10 total measure:", top10["measure_sum"].sum())

Number of rows in top10: 10
Top 10 total measure: 2725281293810


In [19]:
top10.to_csv(
    CONFIG["output_dir"] / "top10.csv",
    index=False,
)

print("top10.csv saved.")

top10.csv saved.


In [20]:
numpy_results = run_numpy_comparison(data)

numpy_results

,method,result,median_time_seconds
0,Python loop,1.627477e+10,0.001977
1,NumPy vectorized,1.627477e+10,0.000136


In [21]:
print("Loop and vectorized results agree:", numpy_results.attrs["numpy_equal"])

Loop and vectorized results agree: True


In [22]:
numpy_results.to_csv(
    CONFIG["output_dir"] / "numpy_comparison.csv",
    index=False,
)

print("numpy_comparison.csv saved.")

numpy_comparison.csv saved.


In [23]:
create_bar_plot(
    top10,
    CONFIG["output_dir"] / "bar.png",
)

print("bar.png created.")

bar.png created.


In [24]:
create_heatmap(
    pivot,
    CONFIG["output_dir"] / "heatmap.png",
)

print("heatmap.png created.")

heatmap.png created.


In [25]:
# Run the updated program to generate consistent final outputs.
from main import main

main()

# Display the complete validation report.
validation_results = pd.read_csv(
    CONFIG["output_dir"] / "validation.csv"
)

print("Validation results:")
display(validation_results)

# Display the expanded audit log.
audit_log = pd.read_csv(
    CONFIG["output_dir"] / "audit_log.csv"
)

print(f"Audit entries: {len(audit_log)}")
display(audit_log.head(6))


Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...
Processing chunk 7...
Processing chunk 8...
Processing chunk 9...
Processing chunk 10...
Processing chunk 11...
Processing chunk 12...
Processing chunk 13...
Processing chunk 14...
Processing chunk 15...
Processing chunk 16...
Processing chunk 17...
Processing chunk 18...
Processing chunk 19...
Processing chunk 20...
Processing chunk 21...
Processing chunk 22...
Processing chunk 23...
All validation checks passed.

Raw rows: 2,236,612
Selected rows: 2,236,612
Excluded rows: 0
Missing numerical values: 0
Invalid numerical values: 0

Summary tables and plots created successfully.

NumPy comparison:
          method       result  median_time_seconds
     Python loop 1.627477e+10             0.001434
NumPy vectorized 1.627477e+10             0.000018
Validation results:


,check,expected,actual,tolerance,pass
0,raw row count vs 2015 reference,2236612,2236612,0.00,True
1,raw measure sum vs 2015 reference,3587267375257,3587267375257.0,1.00,True
2,raw rows = selected rows + excluded rows,2236612,2236612,0.00,True
3,grouped row counts,2236612,2236612,0.00,True
4,grouped measure sum,3587267375257,3587267375257,1.00,True
5,grouped_two measure sum,3587267375257,3587267375257,1.00,True
6,pivot interior measure sum,3587267375257,3587267375257.0,1.00,True
7,bar plot values match top10,True,True,0.00,True
8,heatmap values match pivot interior,True,True,0.00,True
9,loop and NumPy results agree,16274765862.300007,16274765862.3,0.01,True


Audit entries: 150


,step,operation,rule,rows_before,rows_after
0,load,Load chunk 1,Read the original CSV without modifying it,100000,100000
1,inspect,Inspect numerical values,Convert values to numeric for inspection; inva...,100000,100000
2,filter,Apply two-condition filter,tq is not missing AND dutiablevaluephp > 0,100000,100000
3,clean,Prepare selected records,Convert measure to numeric; fill missing count...,100000,100000
4,derive,Create numerical column,dutiablevalue_million_php = dutiablevaluephp /...,100000,100000
5,derive,Create value band,High when million-PHP value >= 100; otherwise ...,100000,100000
